# 17 · Task model v2 (hourly 1 h route weather, thermal sub-models & pilot model)

Upgrades the task model from 5-hour window averages (`task_v1`, notebook 13) to **1-hour space-time along-route weather** (`pta/route_weather.py`, `scripts/task_model_v2.py`):

1. **Hourly (1 h) along-route weather (`M2`):** relative task-hour slots (`h0..h4`), diurnal growth/decay & bottleneck features, space-time ETA-coupled legs (`leader = 28 km/h`, `pack = 18 km/h`), and vertically interpolated flight-altitude wind (`10 m, 100 m, 925..500 hPa`) projected onto each leg at its estimated hour of arrival.
2. **1 km terrain + 0.1° ERA5-Land (`M3`):** route elevation profile (mountain passes), relief, hourly sun-on-slope, lee-side wind share, and 0.1° ERA5-Land surface fluxes (isolated from the forecast-compatible feature set).
3. **Leakage-free in-fold thermal sub-models + MacCready kinematics (`M4`):** `core60_p75_ms`, calibrated ASL ceiling residual, and crossing climb chance at 300 m / 700 m AGL, trained strictly inside each CV fold and combined with leg winds into MacCready speed and time margins per glider class.
4. **Pilot-level model (`M5`):** ~173k launched pilot results (pre-task Elo, strictly prior-competition history, glider class, harness, pilot × weather interactions) aggregated to task goal share and winner speed, plus an in-fold convex blend with `M4`.
5. **Hourly Open-Meteo forecast verification (2022+):** runs leakage-free `task_v2` folds on all 9 archived Open-Meteo models, comparing **raw** vs **chip-hour Quantile Mapping (`hourly_qm`)** vs **task-feature QM (`feature_qm`)** with competition-clustered paired bootstrap tests.

Writes `database/models/task_v2/` and `database/models/forecast_multivar_eval/task_v2/`. Use a **High-RAM** runtime (about 20–45 minutes depending on `THERMAL_ROWS`). Run notebook `07c` with `EVERY = 1` first for native hourly `era5_levels` (missing hours fall back cleanly to linear interpolation).

## 1. Settings

In [ ]:
DRIVE_FOLDER_ID = "1_JO0XMMQ4NKVfpeA8uKQRkZZPNC4sqZ7"   # project folder in Google Drive
THERMAL_ROWS = 800_000   # crossings sample for in-fold climb_prob sub-model (0 to skip)
RUN_FORECAST_V2 = True   # also score hourly Task v2 on 2022+ Open-Meteo forecast caches
FC_THERMAL_ROWS = 400_000
REPO_URL = "https://github.com/RobertojonesFAI/Pre-task-analyzer.git"
REPO = "/content/Pre-task-analyzer"

## 2. Mount Drive, get the code

In [ ]:
import json, os, subprocess
import numpy as np
import pandas as pd
from google.colab import auth, drive
from googleapiclient.discovery import build

auth.authenticate_user()
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")
service = build("drive", "v3")
FIELDS = "id, name, parents, driveId, ownedByMe"


def path_up(file_id, root):
    names, current = [], file_id
    while True:
        meta = service.files().get(fileId=current, fields=FIELDS, supportsAllDrives=True).execute()
        if not meta.get("parents"):
            return os.path.join(root, *reversed(names))
        names.append(meta["name"])
        current = meta["parents"][0]


def drive_path(folder_id):
    meta = service.files().get(fileId=folder_id, fields=FIELDS, supportsAllDrives=True).execute()
    if meta.get("driveId"):
        drive_name = service.drives().get(driveId=meta["driveId"]).execute()["name"]
        return path_up(folder_id, f"/content/drive/Shareddrives/{drive_name}")
    if meta.get("ownedByMe"):
        return path_up(folder_id, "/content/drive/MyDrive")
    page = None
    while True:
        res = service.files().list(
            q="mimeType='application/vnd.google-apps.shortcut' and trashed=false",
            fields="nextPageToken, files(id, shortcutDetails)", pageToken=page).execute()
        for f in res.get("files", []):
            if f.get("shortcutDetails", {}).get("targetId") == folder_id:
                return path_up(f["id"], "/content/drive/MyDrive")
        page = res.get("nextPageToken")
        if not page:
            raise SystemExit(f"'{meta['name']}' is shared with you. Add a shortcut to it in your Drive, then rerun.")


ROOT = drive_path(DRIVE_FOLDER_ID)
DB = os.path.join(ROOT, "database")
assert os.path.isfile(os.path.join(DB, "task_fields.parquet")), "Run notebook 10 first: task_fields.parquet is missing"
subprocess.run(["pip", "install", "-q", "lightgbm"], check=True)
print("Database in", DB)

if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)


def run(*cmd):
    proc = subprocess.Popen(["python3", "-u", *cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        for line in proc.stdout:
            print(line, end="")
        code = proc.wait()
    except KeyboardInterrupt:
        proc.terminate()
        proc.wait()
        print("Stopped.")
        raise
    if code != 0:
        raise RuntimeError(f"{os.path.basename(cmd[0])} stopped with exit code {code}")
    return code

## 3. Pre-flight data check

Checks hourly `era5_levels` coverage and confirms all input tables (`era5_land`, `truth/`, `ratings`, `result_gliders`, `crossings`) are present.

In [ ]:
for rel in [
    "weather/era5_single.parquet", "weather/era5_levels.parquet", "weather/era5_land.parquet",
    "cells.parquet", "ratings.parquet", "result_gliders.parquet", "crossings.parquet",
    "truth/truth_ceiling.parquet", "truth/truth_climb_groups.parquet",
]:
    p = os.path.join(DB, rel)
    print(f"  {rel:38s}: {'OK' if os.path.isfile(p) else 'MISSING'}")

lv_p = os.path.join(DB, "weather", "era5_levels.parquet")
if os.path.isfile(lv_p):
    lv = pd.read_parquet(lv_p, columns=["chip_id", "time"])
    lv["time"] = pd.to_datetime(lv["time"])
    diffs = lv.sort_values(["chip_id", "time"]).groupby("chip_id")["time"].diff().dt.total_seconds() / 3600.0
    wd = diffs[(diffs > 0) & (diffs <= 6)]
    print(f"\nera5_levels: {len(lv):,} rows, within-day median step = {wd.median():.1f} h, share == 1 h: {(wd == 1.0).mean():.1%}")

## 4. Train and score Task model v2 (ablation ladder M0..M5)

Scores every stage on identical 5-fold grouped CV splits (`event_key` = new competition, `site_id` = new site) with competition/site-clustered paired bootstrap 95% CIs against `task_v1`.

In [ ]:
run(f"{REPO}/scripts/task_model_v2.py", DB, "--thermal-rows", str(THERMAL_ROWS))

## 5. Ablation ladder, paired bootstrap & pilot calibration charts

In [ ]:
import matplotlib.pyplot as plt

M2 = os.path.join(DB, "models", "task_v2")
abl = pd.read_csv(os.path.join(M2, "ablation.csv"))
boot = pd.read_csv(os.path.join(M2, "paired_bootstrap.csv"))
imp = pd.read_csv(os.path.join(M2, "importance.csv"))

print("Ablation summary:")
display(abl)
print("\nPaired bootstrap vs M1_task_v1_5h (negative = better than task_v1):")
display(boot[boot["comparison"] == "vs_task_v1"])

ev = abl[abl["split"] == "new competition"].copy()
fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
ax[0].barh(ev["stage"][::-1], ev["winner_time_mae_min"][::-1], color="#2a78d6", height=0.6)
ax[0].set(xlabel="Winner time MAE (min)", title="Ablation: Winner's time (new competition)")
ax[1].barh(ev["stage"][::-1], ev["goal_share_mae_pts"][::-1], color="#cf5222", height=0.6)
ax[1].set(xlabel="Goal share MAE (points)", title="Ablation: Goal share (new competition)")
for a in ax:
    a.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

# Pilot-level calibration (overall and by glider class)
pp_path = os.path.join(M2, "pilot_predictions.parquet")
if os.path.isfile(pp_path):
    pp = pd.read_parquet(pp_path).dropna(subset=["pred_p_goal", "in_goal"])
    pp["bin"] = pd.qcut(pp["pred_p_goal"], 10, duplicates="drop")
    cal = pp.groupby("bin", observed=True).agg(pred=("pred_p_goal", "mean"), obs=("in_goal", "mean"), n=("in_goal", "size"))
    by_cls = pp.groupby("glider_class").agg(n=("in_goal", "size"), pred=("pred_p_goal", "mean"), obs=("in_goal", "mean"))
    print("\nPilot goal calibration by glider class (out-of-fold):")
    display(by_cls.round(3))
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
    ax[0].plot([0, 1], [0, 1], color="#c3c2b7", lw=1)
    ax[0].plot(cal["pred"], cal["obs"], marker="o", color="#2a78d6", lw=2)
    ax[0].set(xlabel="Predicted P(goal)", ylabel="Observed goal rate", title="Pilot-level calibration (OOF)")
    top_imp = imp[imp["target"] == "winner_kmh"].head(15)[::-1]
    ax[1].barh(top_imp["feature"], top_imp["share"], color="#2a78d6", height=0.6)
    ax[1].set(xlabel="Share of gain", title="Top features (winner_kmh)")
    for a in ax:
        a.spines[["top", "right"]].set_visible(False)
    plt.tight_layout(); plt.show()

## 6. Hourly Open-Meteo forecast verification (2022+, Pillar 4b)

Runs leakage-free 5-fold CV of forecast-ready Task v2 on the 9 cached Open-Meteo forecast models, comparing `raw`, `hourly_qm` (chip-hour Quantile Mapping preserving wind direction), and `feature_qm`. Optionally backfill any missing 2022+ route wind chip-hours first using `--backfill-route-wind` (fastest against the local S3 Open-Meteo server on `:18080`).

In [ ]:
if RUN_FORECAST_V2:
    run(
        f"{REPO}/scripts/forecast_multivar_eval.py", DB,
        "--skip-download", "--task-v2-only",
        "--thermal-rows", str(FC_THERMAL_ROWS),
    )
    v2_fc_dir = os.path.join(DB, "models", "forecast_multivar_eval", "task_v2")
    if os.path.isfile(os.path.join(v2_fc_dir, "task_v2_e2e_scores.csv")):
        display(pd.read_csv(os.path.join(v2_fc_dir, "task_v2_e2e_scores.csv")))
        pb2 = pd.read_csv(os.path.join(v2_fc_dir, "task_v2_paired_bootstrap.csv"))
        display(pb2[pb2["comparison"].isin(["blend_mode_vs_raw", "blend_vs_era5"])])